In [ ]:
# PPR annotation pipeline
# note that all 'files' are contained in quotes, so adjust these for your file locations

In [1]:
# packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Bio import SeqIO, AlignIO
import re
import os

In [ ]:
# run in command line
grep -E 'ID' '/home/pmh3ax/start_files/S_conica_v3.gff' > '/home/pmh3ax/pprfinder_runs/c_run_2/Sc3.gff'
# note that this can also be used to filter for specific annotation types before downstream process
# to do this, replace ID with cds, gene, exon, mrna, or another type

In [ ]:
# gffread run in command line cd
./gffread -w '/home/pmh3ax/pprfinder_runs/c_run_2/Sc3exon.fasta' 
-x '/home/pmh3ax/pprfinder_runs/c_run_2/Sc3cds.fasta' 
-y '/home/pmh3ax/pprfinder_runs/c_run_2/Sc3translatecds.fasta' 
-g '/home/pmh3ax/start_files/Sc_v3.fasta' 
-l 279 -u -S '/home/pmh3ax/pprfinder_runs/c_run_2/Sc3.gff'

In [ ]:
# run in command line- hmmer search
export PATH='/home/pmh3ax/hmmer-3.4/src':$PATH
# note pprs.domt is not used downstream
# all_CTD_Smr is the hmm profile provided by Small
hmmsearch --noali -E 0.1 -o pprs.domt --domtblout '/home/pmh3ax/pprfinder_runs/c_run_2/S_conica_v3_TabOut' '/home/pmh3ax/start_files/all_CTD_Smr.hmm' '/home/pmh3ax/pprfinder_runs/c_run_2/Sc3translatecds.fasta'

In [ ]:
# run in command line- julia pprfinder
module load julia
julia
] # enter package manager
add BioSequences FASTX
# backspace to exit package manager
exit()
julia ./PPRfinder/PPRfinder.jl '/home/pmh3ax/pprfinder_runs/c_run_2/Sc3translatecds.fasta' '/home/pmh3ax/pprfinder_runs/c_run_2/S_conica_v3_TabOut'

In [ ]:

# for loading PPR annotations onto gff


In [2]:
def fasta_df(inputfile):
    headers = []
    sequences = []
    currentsequence = ''
    with open(inputfile, 'r') as file:
        for line in file.readlines():
            line = line.strip()
            if line.startswith('>'):
                if currentsequence:
                    sequences.append(currentsequence)
                    currentsequence = ''
                headers.append(line)
            else:
                currentsequence += line
        if currentsequence:
            sequences.append(currentsequence)
    df = pd.DataFrame({'Headers': headers, 'Sequences': sequences})
    return df

In [ ]:
# pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprfinder_runs/c_run_2/S_conica_v3_TabOut.pprs.fa')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [ ]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprfinder_runs/c_run_2/Sc3.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

In [ ]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR']==True]
gffppr.drop('PPR', axis=1)

In [ ]:
# save to gff
gffppr.to_csv('/home/pmh3ax/pprfinder_runs/c_run_2/Sc3pprs.gff', sep='\t', index=False, header=None)

In [ ]:

# before the below step, manually remove unreliable annotations (see notes)
# import as separate tsv/csv/other


In [ ]:
# or if continuous no load, did not manually check
pprs = gffppr
pprs.drop('PPR', axis=1)

In [ ]:
# ppr gff
pprs = pd.read_csv('/home/pmh3ax/pprfinder_runs/c_run_2/Sc3pprs.gff', sep='\t', header=None)
pprs.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID', 'ppr']
# all pprs, safe to drop col
pprs = pprs.drop('ppr', axis=1)

In [ ]:
# excludes utrs, not ppr significant, generally
pprs['type'].value_counts()
pprs = pprs[~pprs['type'].isin(['three_prime_UTR', 'five_prime_UTR'])]

In [ ]:
# manual annotation reduced
annot = pd.read_csv('/home/pmh3ax/pprfinder_runs/c_run_2/S_conica_v3_Annotations.tsv', sep='\t', header=None)
# if different, note and adjust accordingly for match, bounds
annot = annot.drop(annot.columns[7],axis=1)
annot.columns = ['name', 'chr', 'type', 'Start', 'End', 'length', 'direction']

In [ ]:
# name column in the extracted annotation tsv is simply type mrna, exon, etc
annot = annot.drop(0)
annot['type'].value_counts()

In [ ]:
# more filter
annot = annot[~annot['type'].isin(['3\'UTR', '5\'UTR'])]
annot['length'] = annot['length'].astype(int)
# spurious remove
annot = annot[annot['length']>2]

In [ ]:
# merge pprs gff with annot manual by using same annotation bounds- check same count
annot['Start'] = annot['Start'].astype(int)
annot['End'] = annot['End'].astype(int)
pprs_manual = pprs.merge(annot[['chr', 'Start', 'End']], on=['chr', 'Start', 'End'], how='inner')

In [ ]:
# perhaps a few dupes still remain after manual
pprs_manual = pprs_manual.drop_duplicates()

In [ ]:
annot['type'].value_counts()

In [ ]:
pprs_manual['type'].value_counts()

In [ ]:
# export all genes
pprs_manual.to_csv('/home/pmh3ax/pprfinder_runs/c_run_2/Sc3pprs_manual.gff', sep='\t', index=False, header=None)

In [ ]:
# read in pprs_manual
pprs_manual = pd.read_csv('/home/pmh3ax/pprfinder_runs/c_run_2/Sc3pprs_manual.gff', sep='\t', header=None)
pprs_manual.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']

In [ ]:
# motifs structure import (whole protein) to find reliable repeat genes
beads = pd.read_csv('/home/pmh3ax/pprfinder_runs/c_run_2/S_conica_v3_TabOut.beads.txt', sep='\t', header=None)
beads.columns = ['ID', 'length', 'motif_arrangement', 'type', 'score']

In [ ]:
# count column, then filter 3 (each motif 1 rna nt, 1 or 2 motifs not reliable repeat)
beads['motif_count'] = beads['motif_arrangement'].apply(lambda x: sum(c.isalpha() for c in str(x)))
beads = beads[beads['motif_count']>=3]

In [ ]:
beads['motif_clean'] = beads['motif_arrangement'].apply(lambda x: re.sub(r'[^a-zA-Z]', '', x))

In [ ]:
# beads with pprs_manual by clean id without unnecessary info, then filter to genes
pprs_manual['ID_clean'] = pprs_manual['ID'].str.extract(r'ID=([^;]+)')
pprs_gene = pprs_manual[pprs_manual['type']=='mRNA']

In [ ]:
# intersection, only beads
pprs_gene_filtered = pprs_gene[pprs_gene['ID_clean'].isin(beads['ID'])].copy()

In [ ]:
# scratch to determine target
# pprs_gene['ID_clean'].unique() #print all
# find which do not have scaffold_n, target string: Sconica or FUN
# copy
pprs_gene_name = pprs_gene_filtered.copy()

In [ ]:
# standardize names that do not include scaffold
target = 'Sconica'
for index, row in pprs_gene_name.iterrows():
    if target in row['ID_clean']:
        row_old = row['ID_clean']
        pprs_gene_name.at[index, 'ID_clean'] = 'Silene_conica_' + row['chr'] + '_' + row_old

In [ ]:
pprs_gene_name = pprs_gene_name.drop('ID', axis=1)
# add to export pprs_gene
pprs_gene_name.to_csv('/home/pmh3ax/pprfinder_runs/Sc3_pprs_clean.gff', sep='\t', index=False, header=None)

In [ ]:
# export beads for grouping
beads.to_csv('/home/pmh3ax/pprfinder_runs/Sc3_beads_clean.gff', sep='\t', index=False, header=None)

In [ ]:
# or, read in here picking up
# CHANGE FOR SPECIFIC SPECIES

In [ ]:
# motif sort
beadsp = beads[beads['type']=='P']
beadsp_only = beadsp[~beadsp['motif_clean'].str.contains('RFLCTD')]
beadsp_rfl = beadsp[beadsp['motif_clean'].str.contains('RFLCTD')]

In [ ]:
# sort
beadsp_onlys = beadsp_only.sort_values(by='motif_count', ascending=False)
beadsp_rfls = beadsp_rfl.sort_values(by='motif_count',ascending=False)

In [ ]:
# export
beadsp_onlys.to_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_pprs_clean_p_only.gff', sep='\t', index=False, header=None)
beadsp_rfls.to_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_pprs_clean_p_rfl.gff', sep='\t', index=False, header=None)

In [ ]:
# pls
beadspls = beads[beads['type']=='PLS']
beadspls_dyw = beadspls[beadspls['motif_clean'].str.contains('DYW') & ~beadspls['motif_clean'].str.contains('E')]
beadspls_e = beadspls[beadspls['motif_clean'].str.contains('E') & ~beadspls['motif_clean'].str.contains('DYW')]
beadspls_edyw = beadspls[beadspls['motif_clean'].str.contains('DYW') & beadspls['motif_clean'].str.contains('E')]
beadspls_smr = beadspls[beadspls['motif_clean'].str.contains('Smr')]
beadspls_only = beadspls[~beadspls['motif_clean'].str.contains('DYW') & ~beadspls['motif_clean'].str.contains('E') & ~beadspls['motif_clean'].str.contains('Smr')]

In [ ]:
# probably fine to keep with 5 rows, should not have more than this many classes in general
beadspls_onlys = beadspls_only.sort_values(by='motif_count', ascending=False)
beadspls_dyws = beadspls_dyw.sort_values(by='motif_count', ascending=False)
beadspls_es = beadspls_e.sort_values(by='motif_count', ascending=False)
beadspls_edyws = beadspls_edyw.sort_values(by='motif_count', ascending=False)
beadspls_smrs = beadspls_smr.sort_values(by='motif_count', ascending=False)

In [ ]:
# export
beadspls_onlys.to_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_pprs_clean_pls_only.gff', sep='\t', index=False, header=None)
beadspls_dyws.to_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_pprs_clean_pls_dyw.gff', sep='\t', index=False, header=None)
beadspls_es.to_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_pprs_clean_pls_e.gff', sep='\t', index=False, header=None)
beadspls_edyws.to_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_pprs_clean_pls_edyw.gff', sep='\t', index=False, header=None)
beadspls_smrs.to_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_pprs_clean_pls_smr.gff', sep='\t', index=False, header=None)

In [ ]:
def beads_to_pprs_gene(beadsgff, pprs_gene, idname):
    # Filter pprs_gene based on beadsgff IDs
    pprs_gene_filtered = pprs_gene[pprs_gene['ID_clean'].isin(beadsgff['ID'])].copy()
    mask = pprs_gene_filtered['ID_clean'].str.contains('Chr|FUN|Sconica', na=False)
    pprs_gene_filtered.loc[mask, 'ID_clean'] = (
        'Silene_conica_' + pprs_gene_filtered.loc[mask, 'chr'] + '_' + pprs_gene_filtered.loc[mask, 'ID_clean']
    )
    if 'ID' in pprs_gene_filtered.columns:
        pprs_gene_filtered = pprs_gene_filtered.drop(columns=['ID'])
    output_path = os.path.join('/home/pmh3ax/pprfinder_runs', f'Sc3_pprs_clean_{idname}.gff')
    pprs_gene_filtered.to_csv(output_path, sep='\t', index=False, header=None)

In [ ]:
beads_to_pprs_gene(beadsp_onlys, pprs_gene, 'p_only')
beads_to_pprs_gene(beadsp_rfls, pprs_gene, 'p_rfl')

In [ ]:
beads_to_pprs_gene(beadspls_onlys, pprs_gene, 'pls_only')
beads_to_pprs_gene(beadspls_smrs, pprs_gene, 'pls_smr')
beads_to_pprs_gene(beadspls_dyws, pprs_gene, 'pls_dyw')
beads_to_pprs_gene(beadspls_es, pprs_gene, 'pls_e')
beads_to_pprs_gene(beadspls_edyws, pprs_gene, 'pls_edyw')

In [ ]:
# automated fix id
awk 'BEGIN {OFS="\t"} 
    $3 == "mRNA" { $9 = "ID=" $9 } 
    { print }' Sc3_pprs_clean_p_only.gff > Sc3_pprs_clean_p_onlyf.gff

In [ ]:
# extract transcripts from pprs, modify each in command line # CHANGE FOR SPECIFIC SPECIES
# cd gffread
./gffread -w /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/p_only.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/Sc3_pprs_clean_p_onlyf.gff
./gffread -w /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/p_rfl.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/Sc3_pprs_clean_p_rflf.gff
./gffread -w /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/pls_only.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/Sc3_pprs_clean_pls_onlyf.gff
./gffread -w /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/pls_smr.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/Sc3_pprs_clean_pls_smrf.gff
./gffread -w /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/pls_dyw.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/Sc3_pprs_clean_pls_dywf.gff
./gffread -w /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/pls_e.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/Sc3_pprs_clean_pls_ef.gff
./gffread -w /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/pls_edyw.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/gff/Sc3_pprs_clean_pls_edywf.gff

In [ ]:
# align each, then cat
# then run in mafft -merge (realign)
# trim trimal
# tree IQTree bootstrap

In [ ]:
# loading rfl group
beads = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sc3_pprs_beads_rfl.gff', sep='\t', header=None)
gff = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sc3_pprs_clean_rfl.gff', sep='\t', header=None)

In [ ]:
# get rid of few beads 
beads[7] = beads[0].apply(lambda x: gff[8].str.contains(x, na=False).any())
extra_rows = beads[~beads[7]]

In [ ]:
beadsf = beads[beads[7]].drop(columns=[7]).reset_index(drop=True)

In [ ]:
# all beads in gff
extra_rows_check = beadsf[~beadsf[0].apply(lambda x: gff[8].str.contains(x, na=False).any())]
extra_rows_check

In [ ]:
def partial_merge(df1, df2, key_col, full_key_col):
    df1['match'] = df1[key_col].apply(lambda x: df2[df2[full_key_col].str.contains(x, na=False, regex=False)][full_key_col].tolist())
    return df1.explode('match').merge(df2, left_on='match', right_on=full_key_col, how='left').drop(columns=['match'])

# Perform the merge with correct column names
merged_gff = partial_merge(beads, gff, key_col=0, full_key_col=8)

In [ ]:
# Drop unwanted columns
merged_gff.drop(columns=['0_x', '1_x', '3_x', '4_x', '5_x', '7_x'], inplace=True)
remaining_columns = merged_gff.columns.tolist()
new_order = [col for col in remaining_columns if col not in ['2_x', '6_x']] + ['2_x', '6_x']
merged_gff = merged_gff[new_order]
merged_gff

In [ ]:
merged_gff.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3_pprs_beads_rfl.gff', sep='\t', header=None)

In [ ]:
# put on geneious chr view..
# then get allpprs and rf gff for fasta convert

In [ ]:
# Read the input GFF file
input_file = '/home/pmh3ax/pprs_analysis_other/Sc3_pprs_clean.gff'
output_file = '/home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_allpprs.gff'
input_file2 = '/home/pmh3ax/pprs_analysis_other/Sc3_pprs_clean_rfl.gff'
output_file2 = '/home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_rfl.gff'

In [ ]:
with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        if line.strip():  # Check if the line is not empty
            columns = line.strip().split('\t')
            # Add ID= before the last column
            columns[-1] = f'ID={columns[-1]}'
            # Write the modified line to the output file
            outfile.write('\t'.join(columns) + '\n')

In [ ]:
with open(input_file2, 'r') as infile, open(output_file2, 'w') as outfile:
    for line in infile:
        if line.strip():  # Check if the line is not empty
            columns = line.strip().split('\t')
            # Add ID= before the last column
            columns[-1] = f'ID={columns[-1]}'
            # Write the modified line to the output file
            outfile.write('\t'.join(columns) + '\n')

In [ ]:
# to get columns for each, rfl/non
mastergff = pd.read_csv(input_file, sep='\t', header=None)
mastergff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
mastergff

In [ ]:
rflgff = pd.read_csv(input_file2, sep='\t', header=None)
rflgff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
rflgff

In [ ]:
# Undo the standardization by removing 'Silene_conica_<chr>_' if it was added
mastergff['ID_clean'] = mastergff['ID'].str.replace(r'^Silene_conica_Chr\d{2}_', '', regex=True)
rflgff['ID_clean'] = rflgff['ID'].str.replace(r'^Silene_conica_Chr\d{2}_', '', regex=True)

In [ ]:
mastergff['rfl'] = mastergff['ID_clean'].isin(rflgff['ID_clean'])
mastergff['rfl'].value_counts()

In [ ]:
rflgff

In [ ]:
nonrflgff = mastergff[mastergff['rfl']==False]
nonrflgff = nonrflgff.drop(columns=['rfl','ID'], axis=1)
nonrflgff

In [ ]:
rflgff.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3_pprs_clean_rfl.gff', sep='\t', header=None, index=False)

In [ ]:
nonrflgff.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3_pprs_clean_nonrfl.gff', sep='\t', header=None, index=False)

In [ ]:
input_file3 = '/home/pmh3ax/pprs_analysis_other/Sc3_pprs_clean_nonrfl.gff'
output_file3 = '/home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_nonrfl.gff'

In [ ]:
with open(input_file3, 'r') as infile, open(output_file3, 'w') as outfile:
    for line in infile:
        if line.strip():  # Check if the line is not empty
            columns = line.strip().split('\t')
            # Add ID= before the last column
            columns[-1] = f'ID={columns[-1]}'
            # Write the modified line to the output file
            outfile.write('\t'.join(columns) + '\n')

In [ ]:
# gffread to get transcripts sequences
./gffread -w /home/pmh3ax/pprs_analysis_other/Sc3rflseqs.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_rfl.gff
./gffread -w /home/pmh3ax/pprs_analysis_other/Sc3allpprsseqs.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_allpprs.gff
./gffread -w /home/pmh3ax/pprs_analysis_other/Sc3nonrflseqs.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_nonrfl.gff

In [ ]:
# build hmm- requires biopython convert to change afa to sto
AlignIO.convert('/home/pmh3ax/pprs_analysis_other/Sc3trim_p_rfl_aa.fa', "fasta",
                '/home/pmh3ax/pprs_analysis_other/Sc3rfl_aa_hmm.sto', "stockholm")
AlignIO.convert('/home/pmh3ax/pprs_analysis_other/Sc3trim_p_rfl_nt.fa', "fasta",
                '/home/pmh3ax/pprs_analysis_other/Sc3rfl_nt_hmm.sto', "stockholm")

In [ ]:
# hmm path thing
export PATH='/home/pmh3ax/hmmer-3.4/src':$PATH
# build rfl profile
hmmbuild /home/pmh3ax/pprs_analysis_other/Sc3rfl_nt.hmm /home/pmh3ax/pprs_analysis_other/Sc3rfl_nt_hmm.sto
hmmbuild /home/pmh3ax/pprs_analysis_other/Sc3rfl_aa.hmm /home/pmh3ax/pprs_analysis_other/Sc3rfl_aa_hmm.sto

In [ ]:
# run hmmer
# path if not already
nhmmer --noali -E 0.1 -o pprs.domt --tblout /home/pmh3ax/pprs_analysis_other/Sc3nonrflseqsTabOut /home/pmh3ax/pprs_analysis_other/Sc3rfl_nt.hmm /home/pmh3ax/pprs_analysis_other/Sc3nonrflseqs.fa
hmmsearch --noali -E 0.1 -o pprs.domt --domtblout /home/pmh3ax/pprs_analysis_other/Sc3nonrflseqs-translationTabOut /home/pmh3ax/pprs_analysis_other/Sc3rfl_aa.hmm /home/pmh3ax/pprs_analysis_other/Sc3nonrflseqs-translation.fasta

In [ ]:
# intersecting hits from nt and aa hmmsearch
nthmmresult = "/home/pmh3ax/pprs_analysis_other/Sc3nonrflseqsTabOut"
aahmmresult = "/home/pmh3ax/pprs_analysis_other/Sc3nonrflseqs-translationTabOut"
df_nthmm = pd.read_csv(nthmmresult, sep='\s+', header=None)
df_aahmm = pd.read_csv(aahmmresult, sep='\s+', header=None)
df_aahmm[0] = df_aahmm[0].str.replace("_translation", "", regex=False)

# Find common IDs in the first column
common_ids = set(df_nthmm[0]) & set(df_aahmm[0])
df_nthmm_filtered = df_nthmm[df_nthmm[0].isin(common_ids)]
df_aahmm_filtered = df_aahmm[df_aahmm[0].isin(common_ids)]

df_nthmm_filtered.to_csv(nthmmresult + "_filtered", sep=" ", index=False, header=False)
df_aahmm_filtered.to_csv(aahmmresult + "_filtered", sep=" ", index=False, header=False)
print("Filtering complete. Saved filtered files.")

In [ ]:
# match with gff to id seqs
nthmm_filtered = "/home/pmh3ax/pprs_analysis_other/Sc3nonrflseqsTabOut_filtered"
gff_file = "/home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_nonrfl.gff"
output_gff = "/home/pmh3ax/pprs_analysis_other/hmm_id_nonrfl.gff"

df_nthmm = pd.read_csv(nthmm_filtered, sep='\s+', header=None)
valid_ids = set(df_nthmm[0])

gff = pd.read_csv(gff_file, sep="\t", header=None, comment='#', dtype=str)

def id_in_gff(attributes):
    return any(id_ in attributes for id_ in valid_ids)

gff_filtered = gff[gff.iloc[:, 8].apply(id_in_gff)]

# Save filtered GFF
gff_filtered.to_csv(output_gff, sep="\t", index=False, header=False)

print("GFF filtering complete. Saved to:", output_gff)

In [ ]:
# fasta
./gffread -w /home/pmh3ax/pprs_analysis_other/Sc3nonrfl_hmm_seqs.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprs_analysis_other/Sc3hmm_id_nonrfl.gff

In [ ]:
input_file = "/home/pmh3ax/pprs_analysis_other/Sc3rflseqs-translation.fasta"  # Change this to your filename
output_file = "/home/pmh3ax/pprs_analysis_other/Sc3rflseqs-translation_fixed.fasta"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        if line.startswith(">"):
            # Remove the specified prefix
            line = re.sub(r'^>Silene_conica_Chr\d{2}_', '>', line)
            # Truncate to 50 characters
            line = line[:50]
        outfile.write(line)

In [ ]:
input_file2 = "/home/pmh3ax/pprs_analysis_other/Sc3rflseqs.fa"  # Change this to your filename
output_file2 = "/home/pmh3ax/pprs_analysis_other/Sc3rflseqs_fixed.fa"

with open(input_file2, "r") as infile, open(output_file2, "w") as outfile:
    for line in infile:
        if line.startswith(">"):
            # Remove the specified prefix
            line = re.sub(r'^>Silene_conica_Chr\d{2}_', '>', line)
            # Truncate to 50 characters
            line = line[:50]
        outfile.write(line)

In [ ]:
# make blast db, in command line
# module load blast
makeblastdb -in /home/pmh3ax/pprs_analysis_other/blastdb/Sc3rflseqs_fixed.fa -dbtype nucl -parse_seqids
makeblastdb -in /home/pmh3ax/pprs_analysis_other/blastdb/Sc3rflseqs-translation_fixed.fasta -dbtype prot -parse_seqids


In [ ]:
# module if not already
blastp -query /home/pmh3ax/pprs_analysis_other/Sc3nonrflseqs-translation.fasta -db /home/pmh3ax/pprs_analysis_other/blastdb/Sc3rflseqs-translation_fixed.fasta -out /home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-translationblast.output -outfmt 6
blastn -query /home/pmh3ax/pprs_analysis_other/Sc3nonrflseqs.fa -db /home/pmh3ax/pprs_analysis_other/blastdb/Sc3rflseqs_fixed.fa -out /home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-blast.output -outfmt 6


In [ ]:
# blast results
blast_resultsnt = pd.read_csv('/home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-blast.output', sep='\t', header=None)
blast_resultsnt

In [ ]:
blast_resultsnt1 = blast_resultsnt[blast_resultsnt[2]>=95]
blast_resultsnt1

In [ ]:
# blast results
blast_resultsaa = pd.read_csv('/home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-translationblast.output', sep='\t', header=None)
blast_resultsaa

In [ ]:
blast_resultsaa1 = blast_resultsaa[blast_resultsaa[2]>=85]
blast_resultsaa1

In [ ]:
# blast_resultsnt1
# blast_resultsaa1

blast_resultsaa1[0] = blast_resultsaa1[0].str.replace("_translation", "", regex=False)
common_ids = set(blast_resultsnt1[0]) & set(blast_resultsaa1[0])

blast_resultsnt1_f = blast_resultsnt1[blast_resultsnt1[0].isin(common_ids)]
blast_resultsaa1_f = blast_resultsaa1[blast_resultsaa1[0].isin(common_ids)]

blast_resultsnt1_f.to_csv('/home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-blast.output_filtered.txt', sep="\t", index=False, header=False)
blast_resultsaa1_f.to_csv('/home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-translationblast.output_filtered.txt', sep="\t", index=False, header=False)
print("Filtering complete. Saved filtered files.")

In [ ]:
blast_resultsnt1_f_path = '/home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-blast.output_filtered.txt'
blast_resultsaa1_f_path = '/home/pmh3ax/pprs_analysis_other/blastdb/Sc3nonrflseqs-translationblast.output_filtered.txt'

gff_file = '/home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_nonrfl.gff'
output_gff = '/home/pmh3ax/pprs_analysis_other/Sc3blast_id_nonrfl.gff'

# Load the GFF file
gff = pd.read_csv(gff_file, sep="\t", header=None, comment='#', dtype=str)
valid_ids = set(blast_resultsnt1_f[0]).union(set(blast_resultsaa1_f[0]))

def id_in_gff(attributes):
    return any(id_ in attributes for id_ in valid_ids)
gff_filtered = gff[gff.iloc[:, 8].apply(id_in_gff)]

gff_filtered.to_csv(output_gff, sep="\t", index=False, header=False)
print("Filtering complete. Saved filtered files.")

In [ ]:
# fasta
./gffread -w /home/pmh3ax/pprs_analysis_other/Sc3nonrfl_blast_seqs.fa -g /home/pmh3ax/start_files/Sc_v3.fasta /home/pmh3ax/pprs_analysis_other/Sc3blast_id_nonrfl.gff

In [ ]:
# File paths
blast_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3nonrfl_blast_seqs.fa'
hmm_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3nonrfl_hmm_seqs.fa'
combined_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3combined_nonrfl_seqs.fa'

blast_records = SeqIO.to_dict(SeqIO.parse(blast_fasta_path, 'fasta'))
hmm_records = SeqIO.to_dict(SeqIO.parse(hmm_fasta_path, 'fasta'))
combined_records = {**blast_records, **hmm_records}

with open(combined_fasta_path, 'w') as output_handle:
    SeqIO.write(combined_records.values(), output_handle, 'fasta')
print("Combined FASTA file saved to:", combined_fasta_path)

In [ ]:
# File paths- coopt to get all rfls potentially
blast_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3combined_nonrfl_seqs.fa'
hmm_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3rflseqs.fa'
combined_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3combined_rfl_seqs.fa'

blast_records = SeqIO.to_dict(SeqIO.parse(blast_fasta_path, 'fasta'))
hmm_records = SeqIO.to_dict(SeqIO.parse(hmm_fasta_path, 'fasta'))
combined_records = {**blast_records, **hmm_records}

with open(combined_fasta_path, 'w') as output_handle:
    SeqIO.write(combined_records.values(), output_handle, 'fasta')
print("Combined FASTA file saved to:", combined_fasta_path)

In [ ]:
# beads and rfl gff
# combined rfl pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprs_analysis_other/Sc3combined_rfl_seqs.fa')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [ ]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sc3_pprs_clean_allpprs.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

In [ ]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR']==True]
gffppr.drop('PPR', axis=1)

In [ ]:
# save to gff
gffppr.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3combined_rfl_pprs.gff', sep='\t', index=False, header=None)

In [ ]:
beads = pd.read_csv('/home/pmh3ax/pprfinder_runs/species-specific_phylo/conica/Sc3_beads_clean.gff', sep='\t', header=None)
beads

In [ ]:
def partial_merge(df1, df2, key_col, full_key_col):
    df1['match'] = df1[key_col].apply(lambda x: df2[df2[full_key_col].str.contains(x, na=False, regex=False)][full_key_col].tolist())
    return df1.explode('match').merge(df2, left_on='match', right_on=full_key_col, how='left').drop(columns=['match'])

# Perform the merge with correct column names
merged_gff = partial_merge(beads, gffppr, key_col=0, full_key_col='ID')

In [ ]:
# Drop unwanted columns
merged_gff.drop(columns=[0, 1, 3, 4, 5], inplace=True)
remaining_columns = merged_gff.columns.tolist()
new_order = [col for col in remaining_columns if col not in [2, 6]] + [2, 6]
merged_gff = merged_gff[new_order]
merged_gff = merged_gff.dropna(ignore_index=True)
merged_gff

In [ ]:
merged_gff = merged_gff.rename(columns={2: 'motifs', 6: 'motif_clean'})
merged_gff

In [ ]:
merged_gff.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3beads_combined_rfl.gff', sep='\t', header=None)

In [ ]:
merged_gff['motif_clean'].value_counts()

In [30]:
# come back after rfl tree
blast_fasta_path = '/home/pmh3ax/pprs_analysis_other/rfl-tree/Sc3collapse_rfl_seqs.txt'
hmm_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3combined_rfl_seqs.fa'
combined_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sc3condensed_rfl_seqs.fa'

with open(blast_fasta_path) as f:
    target_ids = set(line.strip() for line in f)

hmm_records = SeqIO.to_dict(SeqIO.parse(hmm_fasta_path, 'fasta'))
filtered_records = [hmm_records[seq_id] for seq_id in target_ids if seq_id in hmm_records]

with open(combined_fasta_path, 'w') as output_handle:
    SeqIO.write(filtered_records, output_handle, 'fasta')

print(f"Filtered {len(filtered_records)} unique sequences from full-length data.")
print("Filtered FASTA file saved to:", combined_fasta_path)

Filtered 149 unique sequences from full-length data.
Filtered FASTA file saved to: /home/pmh3ax/pprs_analysis_other/Sc3condensed_rfl_seqs.fa


In [ ]:
# run tree build on combined seqs

# PICKUP HERE


In [4]:
# observe rfl/new rfl in tree
file1 = "/home/pmh3ax/pprs_analysis_other/rfl-tree/Sc3trim_rfl_seqs.fa"
file2 = "/home/pmh3ax/pprs_analysis_other/Sc3rflseqs.fa"
tree_file = "/home/pmh3ax/pprs_analysis_other/rfl-tree/Sc3trim_rfl_seqs.fa.contree"

headers1 = {record.id for record in SeqIO.parse(file1, "fasta")}
headers2 = {record.id for record in SeqIO.parse(file2, "fasta")}

common_headers = headers1 & headers2

print(f"Total headers in file1: {len(headers1)}")
print(f"Total headers in file2: {len(headers2)}")
print(f"Common headers found: {len(common_headers)}")

def modify_header(record):
    if record.id in common_headers:
        if "Silene" in record.id:
            record.id = record.id.replace("Silene", "RFL_Silene")
            record.description = record.id
    return record

records1 = (modify_header(record) for record in SeqIO.parse(file1, "fasta"))
output_file1 = file1.replace(".fa", "_modified.fa")
SeqIO.write(records1, output_file1, "fasta")
print(f"Updated trimmed FASTA file saved as: {output_file1}")

modified_headers = {record.id for record in SeqIO.parse(output_file1, "fasta")}

with open(tree_file, "r") as f:
    tree_data = f.read()

for header in modified_headers:
    original_header = header.replace("RFL_Silene", "Silene")
    if original_header in tree_data:
        tree_data = tree_data.replace(original_header, header)

output_tree_file = tree_file.replace(".contree", "_modified.contree")
with open(output_tree_file, "w") as f:
    f.write(tree_data)

print(f"Updated tree file saved as: {output_tree_file}")

Total headers in file1: 238
Total headers in file2: 53
Common headers found: 53
Updated trimmed FASTA file saved as: /home/pmh3ax/pprs_analysis_other/rfl-tree/Sc3trim_rfl_seqs_modified.fa
Updated tree file saved as: /home/pmh3ax/pprs_analysis_other/rfl-tree/Sc3trim_rfl_seqs.fa_modified.contree


In [3]:
# after tree vis, can filter out non-rfl clade entirely
# rest of rfls and newly id hits are justifiable
def df_to_fasta(df, output_file):
    with open(output_file, 'w') as f:
        for _, row in df.iterrows():
            f.write(f"{row['Headers']}\n{row['Sequences']}\n")

In [13]:
file_large = '/home/pmh3ax/pprs_analysis_other/rfl-tree/Sc3trim_rfl_seqs_modified.fa'
file_small = '/home/pmh3ax/pprs_analysis_other/rfl-tree/Sc3seqsfa-extract.fasta'
dfl = fasta_df(file_large)
dfs = fasta_df(file_small)

headers_to_remove = dfs['Headers'].tolist()
dfl_filtered = dfl[dfl['Headers'].isin(headers_to_remove)]

dfl_filtered['Headers'] = dfl_filtered['Headers'].str.replace(r'(.*RFL_)', r'>', regex=True)

output_file = '/home/pmh3ax/pprs_analysis_other/Sc3trimmed_rfl_seqs.fasta'
df_to_fasta(dfl_filtered, output_file)

/tmp/ipykernel_517960/3738112415.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfl_filtered['Headers'] = dfl_filtered['Headers'].str.replace(r'(.*RFL_)', r'>', regex=True)


In [25]:
file_large = '/home/pmh3ax/pprs_analysis_other/Sc3allpprsseqs.fa'
file_small = '/home/pmh3ax/pprs_analysis_other/Sc3trimmed_rfl_seqs.fasta'

dfl = fasta_df(file_large)
dfs = fasta_df(file_small)

dfl['Headers'] = dfl['Headers'].str.strip()
dfs['Headers'] = dfs['Headers'].str.strip().str.lstrip('>')

headers_to_keep = set(dfs['Headers'])
dfl_filtered = dfl[dfl['Headers'].str.contains('|'.join(headers_to_keep), na=False)]

output_file = '/home/pmh3ax/pprs_analysis_other/Sc3update_rfl_seqs.fasta'
df_to_fasta(dfl_filtered, output_file)

In [26]:
# run fasta df
# pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprs_analysis_other/Sc3update_rfl_seqs.fasta')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [27]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sc3_pprs_id_allpprs.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

,chr,source,type,Start,End,.,Strand,score,ID
0,Chr01,AUGUSTUS,mRNA,1423577,1425385,.,-,.,ID=Chr01-anno1.g58205.t1
1,Chr01,AUGUSTUS,mRNA,18783480,18786478,.,+,.,ID=Chr01-anno1.g59322.t1
2,Chr01,AUGUSTUS,mRNA,42384860,42395020,.,-,.,ID=Chr01-anno2.g46480.t1
3,Chr01,GeneMark.hmm,mRNA,49839453,49851247,.,-,.,ID=Chr01-long_reads1.PB.110.1
4,Chr01,funannotate,mRNA,52888790,52894086,.,+,.,ID=Silene_conica_Chr01_FUN_002196-T1
...,...,...,...,...,...,...,...,...,...
506,Chr10,AUGUSTUS,mRNA,69211088,69212852,.,-,.,ID=Chr10-anno1.g32664.t1
507,Chr10,maker,mRNA,69215411,69219220,11986,-,.,ID=Silene_conica_Chr10_Sconica_v3_58276-RA
508,Chr10,AUGUSTUS,mRNA,69629146,69631326,.,-,.,ID=Chr10-anno1.g32718.t1
509,Chr10,maker,mRNA,69901700,69904481,.,-,.,ID=Silene_conica_Chr10_Sconica_v3_58367-RA


In [28]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR']==True]
gffppr.drop('PPR', axis=1)

,chr,source,type,Start,End,.,Strand,score,ID
21,Chr01,funannotate,mRNA,83668296,83671205,.,-,.,ID=Silene_conica_Chr01_FUN_004020-T1
22,Chr01,funannotate,mRNA,83675277,83676383,.,-,.,ID=Silene_conica_Chr01_FUN_004022-T1
23,Chr01,GeneMark.hmm,mRNA,83687958,83690135,.,-,.,ID=Chr01-long_reads1.PB.278.1
31,Chr01,AUGUSTUS,mRNA,94039203,94040249,.,+,.,ID=Chr01-anno1.g64107.t1
147,Chr04,funannotate,mRNA,6970882,6974495,.,+,.,ID=Silene_conica_Chr04_FUN_018466-T1
...,...,...,...,...,...,...,...,...,...
492,Chr10,AUGUSTUS,mRNA,64093285,64094934,.,-,.,ID=Chr10-anno2.g36369.t1
493,Chr10,AUGUSTUS,mRNA,64369375,64370871,.,+,.,ID=Chr10-anno2.g36403.t1
495,Chr10,maker,mRNA,64682229,64684011,.,+,.,ID=Silene_conica_Chr10_Sconica_v3_57642-RA
505,Chr10,maker,mRNA,69202055,69206725,12095,-,.,ID=Silene_conica_Chr10_Sconica_v3_58272-RA


In [29]:
# save to gff
gffppr.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3realign_rfl.gff', sep='\t', index=False, header=None)

In [ ]:
# get gff, run realign rfl, run realign all then region for all, blast
# don't forget to trim when realigning (after geneious mafft local)

In [62]:
# checking here
# compare rflctd-id pprs with new id pprs
fa_rfl = fasta_df('/home/pmh3ax/pprs_analysis_other/Sc3rflseqs.fa')
fa_new = fasta_df('/home/pmh3ax/pprs_analysis_other/Sc3update_rfl_seqs.fasta')

In [ ]:
merged_fa = pd.concat([fa_rfl, fa_new], axis=1)
merged_fa

In [54]:
merged_fa.to_csv('/home/pmh3ax/pprs_analysis_other/merged_fa_check.csv', index=False)

In [65]:
# match- change to get rid of first
fa_rfl['Headers'] = fa_rfl['Headers'].apply(lambda x: re.sub(r"(_Chr\d+)(.*?)\1", r"\1\2", x))
fa_new['Headers'] = fa_new['Headers'].apply(lambda x: re.sub(r"(_Chr\d+)(.*?)\1", r"\1\2", x))

In [66]:
fa_rfl['Headers']
fa_new['Headers']

0           >Silene_conica_Chr01_FUN_004020-T1
1           >Silene_conica_Chr01_FUN_004022-T1
2                  >Chr01-long_reads1.PB.278.1
3                       >Chr01-anno1.g64107.t1
4           >Silene_conica_Chr04_FUN_018466-T1
                        ...                   
76                      >Chr10-anno2.g36369.t1
77                      >Chr10-anno2.g36403.t1
78    >Silene_conica_Chr10_Sconica_v3_57642-RA
79    >Silene_conica_Chr10_Sconica_v3_58272-RA
80    >Silene_conica_Chr10_Sconica_v3_58276-RA
Name: Headers, Length: 81, dtype: object

In [67]:
# name
fa_new['rflctdHeaders'] = fa_new['Headers']
fa_new['rflctdHeaders'] = fa_new['Headers'].apply(lambda x: str(x) + 'rfl' if x in fa_rfl['Headers'].values else x)

In [68]:
# count_rfl = fa_new['rflctdHeaders'].str.contains('rfl').sum()
# count_rfl
fa_new

,Headers,Sequences,rflctdHeaders
0,>Silene_conica_Chr01_FUN_004020-T1,ATGTCTTCGCCTTCTTTTCGATTGCGATTACGATTTCAATTTCATC...,>Silene_conica_Chr01_FUN_004020-T1
1,>Silene_conica_Chr01_FUN_004022-T1,ATGATTATGTCTTCGCCTTCTTTTCGATTGCGATTACGATTTCAAT...,>Silene_conica_Chr01_FUN_004022-T1
2,>Chr01-long_reads1.PB.278.1,ATGTCTTCGTCTTCTTTCCGATTGCGATTTCAATTTCAATTGCATT...,>Chr01-long_reads1.PB.278.1
3,>Chr01-anno1.g64107.t1,ATGGCTTTCATGACAGCGCCAACTTTGAGAGCTCATCATCTTCATC...,>Chr01-anno1.g64107.t1
4,>Silene_conica_Chr04_FUN_018466-T1,ATGATGTCTGTTGTTCCCCTACCTTCGATTGCCGAATTTAGCTTGC...,>Silene_conica_Chr04_FUN_018466-T1
...,...,...,...
76,>Chr10-anno2.g36369.t1,ATGTGTTCCCCAATTTTGAAATTAGGGTTTCGATTTCAACATCATT...,>Chr10-anno2.g36369.t1
77,>Chr10-anno2.g36403.t1,ATGCCTTCGATTTTTCGATTTCGATTTCAATTTCAACCTATTTCCC...,>Chr10-anno2.g36403.t1
78,>Silene_conica_Chr10_Sconica_v3_57642-RA,ATGTCTTCCCTTTTTTCTAAATTAGGGTTTCGATTTCAATTTAATT...,>Silene_conica_Chr10_Sconica_v3_57642-RArfl
79,>Silene_conica_Chr10_Sconica_v3_58272-RA,TTTTCCAAGCCATTCATCTTCGTCATCTTCGGTCGCATTGCTGCAT...,>Silene_conica_Chr10_Sconica_v3_58272-RArfl


In [69]:
# Drop the existing 'Headers' column
fa_new.drop(columns=['Headers'], inplace=True)
# Move 'rflctdHeaders' to the first column and rename it to 'Headers'
fa_new.insert(0, 'Headers', fa_new.pop('rflctdHeaders'))
fa_new

,Headers,Sequences
0,>Silene_conica_Chr01_FUN_004020-T1,ATGTCTTCGCCTTCTTTTCGATTGCGATTACGATTTCAATTTCATC...
1,>Silene_conica_Chr01_FUN_004022-T1,ATGATTATGTCTTCGCCTTCTTTTCGATTGCGATTACGATTTCAAT...
2,>Chr01-long_reads1.PB.278.1,ATGTCTTCGTCTTCTTTCCGATTGCGATTTCAATTTCAATTGCATT...
3,>Chr01-anno1.g64107.t1,ATGGCTTTCATGACAGCGCCAACTTTGAGAGCTCATCATCTTCATC...
4,>Silene_conica_Chr04_FUN_018466-T1,ATGATGTCTGTTGTTCCCCTACCTTCGATTGCCGAATTTAGCTTGC...
...,...,...
76,>Chr10-anno2.g36369.t1,ATGTGTTCCCCAATTTTGAAATTAGGGTTTCGATTTCAACATCATT...
77,>Chr10-anno2.g36403.t1,ATGCCTTCGATTTTTCGATTTCGATTTCAATTTCAACCTATTTCCC...
78,>Silene_conica_Chr10_Sconica_v3_57642-RArfl,ATGTCTTCCCTTTTTTCTAAATTAGGGTTTCGATTTCAATTTAATT...
79,>Silene_conica_Chr10_Sconica_v3_58272-RArfl,TTTTCCAAGCCATTCATCTTCGTCATCTTCGGTCGCATTGCTGCAT...


In [70]:
df_to_fasta(fa_new, '/home/pmh3ax/pprs_analysis_other/Sc3marked_rfl_seqs.fasta')

In [71]:
align = fasta_df('/home/pmh3ax/pprs_analysis_other/step2_realign/Sc3realign_rfl_seqs-out.fa')

In [72]:
# Assuming fa_new has headers in a column named 'header' and align has sequences in a column named 'sequence'
merged_df = fa_new[['Headers']].join(align[['Sequences']]).dropna()
merged_df

,Headers,Sequences
0,>Silene_conica_Chr01_FUN_004020-T1,----------------------------------------------...
1,>Silene_conica_Chr01_FUN_004022-T1,----------------------------------------------...
2,>Chr01-long_reads1.PB.278.1,----------------------------------------------...
3,>Chr01-anno1.g64107.t1,----------------------------------------------...
4,>Silene_conica_Chr04_FUN_018466-T1,----------------------------------------------...
...,...,...
75,>Silene_conica_Chr10_Sconica_v3_57544-RArfl,----------------------------------------------...
76,>Chr10-anno2.g36369.t1,----------------------------------------------...
77,>Chr10-anno2.g36403.t1,----------------------------------------------...
78,>Silene_conica_Chr10_Sconica_v3_57642-RArfl,---------------------------cctaattttctccttcttc...


In [73]:
count_rfl = merged_df['Headers'].str.contains('rfl').sum()
count_rfl

34

In [74]:
df_to_fasta(merged_df, '/home/pmh3ax/pprs_analysis_other/Sc3marked_rfl_seqs_align.fasta')

In [ ]:
# copy for un-marking for alignment, tree again

In [12]:
# checking here
# compare rflctd-id pprs with new id pprs
fa_mark = fasta_df('/home/pmh3ax/pprs_analysis/marked_realign_all_seqs.fa_extraction.fasta')

In [13]:
# name
fa_mark['rflctdHeaders'] = fa_mark['Headers']
fa_mark['rflctdHeaders'] = fa_mark['Headers'].apply(lambda x: x.replace('rfl', '') if 'rfl' in x else x)

In [14]:
#count_rfl = fa_mark['rflctdHeaders'].str.contains('rfl').sum()
#count_rfl
fa_mark = fa_mark.iloc[1:]
fa_mark

,Headers,Sequences,rflctdHeaders
1,>Silene_vulgaris_scaffold_1_003467.1,CTTCATTTCACATCTGCAAATCACTCGCCATTTCTACAATTACATT...,>Silene_vulgaris_scaffold_1_003467.1
2,>Silene_vulgaris_scaffold_5_FUN_025820-T1,ATGTTTTCCCCTATTTCTAATCTAGGGTTTCGATTTCAACTTTATT...,>Silene_vulgaris_scaffold_5_FUN_025820-T1
3,>Silene_vulgaris_scaffold_12_FUN_053677-T1,ATGTTTCCCCCTTTTTCTAAATTAGGGTTTCGATCTCAATCTCATT...,>Silene_vulgaris_scaffold_12_FUN_053677-T1
4,>Silene_vulgaris_scaffold_5_001830.1,CATCAAAATGACTAACCCCTAATTCGCCTTTCCATTTCTTGTTCTA...,>Silene_vulgaris_scaffold_5_001830.1
5,>Silene_vulgaris_scaffold_1_002736.1,TAAAAATTGATGTTCTTAGCAGATGGCCATCTCAGTTGTGCAAATC...,>Silene_vulgaris_scaffold_1_002736.1
...,...,...,...
141,>Silene_vulgaris_scaffold_7_FUN_043784-T1rfl,ATGGCGCGAATCATCGGAGTAGGTAACCACCTTGTTCGCCTTTATT...,>Silene_vulgaris_scaffold_7_FUN_043784-T1
142,>Silene_vulgaris_scaffold_7_002479.1rfl,CTTCAGAACAAAAATACAAAACACAGAATCCCCCTTTCTTCTTACT...,>Silene_vulgaris_scaffold_7_002479.1
143,>Silene_vulgaris_scaffold_7_002464.1rfl,CCCAAATTTGTTAACCCTAATTACTCTGGTCGACGATTAACACGGC...,>Silene_vulgaris_scaffold_7_002464.1
144,>Silene_vulgaris_scaffold_7_002472.1rfl,CATCATAGCAATAATACTCATACTCCCTTGTATTAATTCTGCCGAT...,>Silene_vulgaris_scaffold_7_002472.1


In [15]:
# Drop the existing 'Headers' column
fa_mark.drop(columns=['Headers'], inplace=True)
# Move 'rflctdHeaders' to the first column and rename it to 'Headers'
fa_mark.insert(0, 'Headers', fa_mark.pop('rflctdHeaders'))
fa_mark

,Headers,Sequences
1,>Silene_vulgaris_scaffold_1_003467.1,CTTCATTTCACATCTGCAAATCACTCGCCATTTCTACAATTACATT...
2,>Silene_vulgaris_scaffold_5_FUN_025820-T1,ATGTTTTCCCCTATTTCTAATCTAGGGTTTCGATTTCAACTTTATT...
3,>Silene_vulgaris_scaffold_12_FUN_053677-T1,ATGTTTCCCCCTTTTTCTAAATTAGGGTTTCGATCTCAATCTCATT...
4,>Silene_vulgaris_scaffold_5_001830.1,CATCAAAATGACTAACCCCTAATTCGCCTTTCCATTTCTTGTTCTA...
5,>Silene_vulgaris_scaffold_1_002736.1,TAAAAATTGATGTTCTTAGCAGATGGCCATCTCAGTTGTGCAAATC...
...,...,...
141,>Silene_vulgaris_scaffold_7_FUN_043784-T1,ATGGCGCGAATCATCGGAGTAGGTAACCACCTTGTTCGCCTTTATT...
142,>Silene_vulgaris_scaffold_7_002479.1,CTTCAGAACAAAAATACAAAACACAGAATCCCCCTTTCTTCTTACT...
143,>Silene_vulgaris_scaffold_7_002464.1,CCCAAATTTGTTAACCCTAATTACTCTGGTCGACGATTAACACGGC...
144,>Silene_vulgaris_scaffold_7_002472.1,CATCATAGCAATAATACTCATACTCCCTTGTATTAATTCTGCCGAT...


In [17]:
df_to_fasta(fa_mark, '/home/pmh3ax/pprs_analysis/Sv3_last_run_rfl_seqs.fasta')

In [47]:
align = fasta_df('/home/pmh3ax/pprs_analysis/realign_rfl_seqs.fa')

In [53]:
# Assuming fa_new has headers in a column named 'header' and align has sequences in a column named 'sequence'
merged_df = fa_new[['Headers']].join(align[['Sequences']])
merged_df

,Headers,Sequences
0,>Silene_vulgaris_scaffold_1_003467.1,----------------------------------------------...
1,>Silene_vulgaris_scaffold_1_002736.1,----------------------------------------------...
2,>Silene_vulgaris_scaffold_2_003837.1,----------------------------------------------...
3,>Silene_vulgaris_scaffold_5_FUN_025820-T1,----------------------------------------------...
4,>Silene_vulgaris_scaffold_5_001830.1,----------------------------------------------...
...,...,...
151,>Silene_vulgaris_scaffold_13_FUN_054347-T1,----------------------------------------------...
152,>Silene_vulgaris_scaffold_16_FUN_068604-T1,----------------------------------------------...
153,>Silene_vulgaris_scaffold_16_FUN_068798-T1,tgcgtgattttagaaacattagttgcctttttgat-----------...
154,>Silene_vulgaris_scaffold_16_FUN_068800-T1,----------------------------------------------...


In [54]:
count_rfl = merged_df['Headers'].str.contains('rfl').sum()
count_rfl

98

In [55]:
df_to_fasta(merged_df, '/home/pmh3ax/pprs_analysis/marked_rfl_seqs_align.fasta')

In [7]:
# run fasta df
# pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprs_analysis/update_rfl_seqs.fasta')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [ ]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprs_analysis/Sv3_pprs_id_allpprs.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

In [ ]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR']==True]
gffppr.drop('PPR', axis=1)

In [ ]:
# save to gff- can now see on chr
gffppr.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3realign_rfl.gff', sep='\t', index=False, header=None)

In [ ]:
"""
update for existing align, tree
tree in ete-blast
align all
update name
RETRIEVE gff with rfl or not, to id possible fakes...
"""

In [ ]:
# ignore below for now
# for next, will want to intersect to find ppr proteins for each ppr grouped
# then extract seqs for each specified ppr protein
# assess if will align better roughly, adjust gap penalty, align, check...
#

In [ ]:
# df to match, need to define fasta_df function before
vulg_motif = fasta_df('/home/pmh3ax/pprfinder_runs/treepipe/vulgaris/trim_vulgaris.fa')
vulg_motif['Headers'] = vulg_motif['Headers'].str.slice(start=1)

In [ ]:
def scaffold_header(fa_path, headers, loc_names, headers_new, output_path=None):
    # fasta file, headers column title, new fasta with updated names, new headers column title
    # initialize fasta
    fa = fasta_df(fa_path)
    # empty column for match
    fa['new']=''
    # init names
    loc = pd.read_csv(loc_names, sep='\t', header=None, names=['old_name', headers_new])
    loc_list = loc[headers_new].tolist()
    # Iterate through the FASTA DataFrame and update the headers
    for index, row in fa.iterrows():
        old_header = row[headers][1:]
        matching_new_header = next((new_header for new_header in loc_list if old_header in new_header), None)
        fa.at[index, 'new'] = matching_new_header if matching_new_header else old_header
    fa = fa.drop(headers, axis=1)
    fa = fa[['new','Sequences']]
    if output_path:
        fa.to_csv(output_path, sep='\t', index=False, header=None)
    return fa

In [ ]:
# function run- motif
scaffold_header('/home/pmh3ax/pprfinder_runs/treepipe(motif)/vulgaris/trim_vulgaris.fa', 'Headers',
                '/home/pmh3ax/pprfinder_runs/Sv3pprs_gene_scaf.gff', 'ID_clean',
                '/home/pmh3ax/pprfinder_runs/treepipe(motif)/vulgaris/name_vulgaris.fa')

In [ ]:
# function run- nt
scaffold_header('/home/pmh3ax/pprfinder_runs/treetranscript(nt)/vulgaris/trimt_vulgaris.fa', 'Headers',
                '/home/pmh3ax/pprfinder_runs/Sv3pprs_gene_scaf.gff', 'ID_clean',
                '/home/pmh3ax/pprfinder_runs/treetranscript(nt)/vulgaris/namet_vulgaris.fa')

In [ ]:
pprs_gene_filtered.to_csv('/home/pmh3ax/pprfinder_runs/v_run_2/Sv3pprs_gene.gff', sep='\t', index=False, header=None)

In [ ]:
# extract transcripts from pprs genes beads filtered
# cd gffread
./gffread -w /home/pmh3ax/pprfinder_runs/transcripts_vulgaris.fa -g /home/pmh3ax/start_files/S_vulgaris_v3.fasta /home/pmh3ax/pprfinder_runs/v_run_2/Sv3pprs_gene.gff

In [ ]:
# Filter beads to include only rows matching the filtered pprs_gene IDs
filtered_beads = beads[beads['ID'].isin(pprs_gene_filtered['ID_clean'])]
filtered_beads

# Extract the motif_clean column for the matching rows
motif_clean_filtered = filtered_beads['motif_clean']
filtered_beads

In [ ]:
# Specify the output FASTA file
output_fasta = "/home/pmh3ax/pprfinder_runs/motif_vulgaris.fa"

# Open the file in write mode
with open(output_fasta, 'w') as fasta_file:
    for _, row in filtered_beads.iterrows():
        # Write each ID_clean as the header and motif_clean as the sequence
        fasta_file.write(f">{row['ID']}\n{row['motif_clean']}\n")

print(f"FASTA file saved to {output_fasta}")

In [ ]:
# see jobs, mafft to align
# then fast tree and manual correct

In [ ]:

# please note these last few steps are still in refinement


In [ ]:
# run in command line
# using p which contains p (and pls), filter fastas to make 4 separate gene, exon, cds, mrna
grep -E 'gene' '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_out.gff' > '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_gene.gff'

In [ ]:
# starting gff
gff = pd.read_csv('/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type_pprs', 'Start', 'End', 'score_pprs', 'Strand', 'frame', 'ID', 'length', 'motif_arrangement', 'type_beads', 'score_beads', 'motif_count']

In [ ]:
gff = gff.drop(columns=['length', 'motif_arrangement', 'type_beads', 'score_beads', 'motif_count'])

In [ ]:
updated_ids = []
for ID in gff['ID']:
    if ';' in ID:
        ID = ID.split(";")[0]
    if '=' in ID:
        ID = ID.split("=")[1]
    updated_ids.append(ID)
gff['ID'] = updated_ids
gff['type_pprs'] = gff['ID']

In [ ]:
# save to gff
gff.to_csv('/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.gff', sep='\t', index=False, header=None)

In [ ]:
# run in command line- get fasta seqs
# update your version for module
module load bedtools
bedtools getfasta -nameOnly -fo '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.fasta' -fi '/home/pmh3ax/start_files/S_vulgaris_v3.fasta' -bed '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.gff'

In [ ]:
# cd hit to cluster and reduce sequence redundancy- don't use 
# command line
cdhit/bin/cd-hit -i pprfinder_runs/all_run/Sv3_pprs_exon.fasta -o pprfinder_runs/all_run/sv3_pprs_exon_hit

In [ ]:
# small break to get broken-down groups for better aligning

beadsp_onlys['motif_count'].value_counts().sort_index().plot.bar()

In [ ]:
def split_by_frequency_quartiles(df, column):
    # df (pd.DataFrame): The input DataFrame.
    # column (str): The column to calculate frequencies and quartiles.
    df = df.copy()
    df['quartile_group'] = pd.qcut(df[column], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print(df[[column, 'frequency', 'quartile_group']])
    q1 = df[df['quartile_group'] == 'Q1']
    q2 = df[df['quartile_group'] == 'Q2']
    q3 = df[df['quartile_group'] == 'Q3']
    q4 = df[df['quartile_group'] == 'Q4']

    return q1, q2, q3, q4

beadsp_onlys1, beadsp_onlys2, beadsp_onlys3, beadsp_onlys4 = split_by_frequency_quartiles(beadsp_onlys, 'motif_count')

In [ ]:
beadsp_onlys1.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only1.gff', sep='\t', index=False, header=None)
beadsp_onlys2.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only2.gff', sep='\t', index=False, header=None)
beadsp_onlys3.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only3.gff', sep='\t', index=False, header=None)
beadsp_onlys4.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only4.gff', sep='\t', index=False, header=None)

In [ ]:
# small break to get broken-down groups for better aligning

beadspls_onlys['motif_count'].value_counts().sort_index().plot.bar()

In [ ]:
def split_by_frequency_quartiles(df, column):
    # df (pd.DataFrame): The input DataFrame.
    # column (str): The column to calculate frequencies and quartiles.
    df = df.copy()
    df['quartile_group'] = pd.qcut(df[column], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print(df[[column,'quartile_group']])
    q1 = df[df['quartile_group'] == 'Q1']
    q2 = df[df['quartile_group'] == 'Q2']
    q3 = df[df['quartile_group'] == 'Q3']
    q4 = df[df['quartile_group'] == 'Q4']

    return q1, q2, q3, q4

beadspls_onlys1, beadspls_onlys2, beadspls_onlys3, beadspls_onlys4 = split_by_frequency_quartiles(beadspls_onlys, 'motif_count')

In [ ]:
beadspls_onlys1.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only1.gff', sep='\t', index=False, header=None)
beadspls_onlys2.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only2.gff', sep='\t', index=False, header=None)
beadspls_onlys3.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only3.gff', sep='\t', index=False, header=None)
beadspls_onlys4.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only4.gff', sep='\t', index=False, header=None)

In [ ]:
# small break to get broken-down groups for better aligning

beadspls_es['motif_count'].value_counts().sort_index().plot.bar()

In [ ]:
def split_by_frequency_quartiles(df, column):
    # df (pd.DataFrame): The input DataFrame.
    # column (str): The column to calculate frequencies and quartiles.
    df = df.copy()
    df['quartile_group'] = pd.qcut(df[column], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print(df[[column,'quartile_group']])
    q1 = df[df['quartile_group'] == 'Q1']
    q2 = df[df['quartile_group'] == 'Q2']
    q3 = df[df['quartile_group'] == 'Q3']
    q4 = df[df['quartile_group'] == 'Q4']

    return q1, q2, q3, q4

beadspls_es1, beadspls_es2, beadspls_es3, beadspls_es4 = split_by_frequency_quartiles(beadspls_es, 'motif_count')

In [ ]:
beadspls_es1.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e1.gff', sep='\t', index=False, header=None)
beadspls_es2.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e2.gff', sep='\t', index=False, header=None)
beadspls_es3.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e3.gff', sep='\t', index=False, header=None)
beadspls_es4.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e4.gff', sep='\t', index=False, header=None)